# Document pre-processing and feature extraction

Based on NLP2601-DataIngestion

This notebook has developed by members of the [Language Technologies Lab](https://nlp.unibo.it), as a learning resource for the [Natural Language Processing course](https://www.unibo.it/en/study/phd-professional-masters-specialisation-schools-and-other-programmes/course-unit-catalogue/course-unit/2026/530664) of the [International Master's Degree in Artificial Intelligence](https://corsi.unibo.it/2cycle/artificial-intelligence/) of the [University of Bologna](https://www.unibo.it/en/homepage). It is distributed on [Virtuale](https://virtuale.unibo.it/) and can be downloaded personal use only. Any sharing or distribution of the notebook or its adaptations without written consent of the authors is forbidden.

Authors:
- Federico Ruggeri
- Andrea Galassi
- Eleonora Mancini
- Paolo Torroni

In [1]:
# data and numerical management
import pandas as pd
import numpy as np

# progress bar
from tqdm import tqdm

# saving
import simplejson as sj
from pathlib import Path

# vocabulary creation
from typing import List, Callable, Dict
from collections import OrderedDict
from functools import reduce

# pre-processing
import re

In [2]:
local_folder = Path('/Users/paolo/PycharmProjects')
data_folder = local_folder.joinpath('Data')
print(f"Datasets directory: {data_folder}")

dataset_name = "aclImdb"
dataset_path = data_folder.joinpath(dataset_name)
dataset_folder = dataset_path.joinpath("Dataset")

dataframe_folder = dataset_path.joinpath("Dataframe")
dataframe_path = dataframe_folder.joinpath(dataset_name + ".pkl")

df = pd.DataFrame()

if not dataframe_path.exists():
    print(f"Didn't find {dataframe_path}")
    print("Run NLP2601-DataIngestion first to read dataset and build dataframe")
    exit()
else:
    print(f"Recovering dataframe from pickle file {dataframe_path}")
    df=pd.read_pickle(dataframe_path)

preprocessing_folder = dataset_path.joinpath("Preprocessing")
print(f"Preprocessing directory: {preprocessing_folder}")

if not preprocessing_folder.exists():
    preprocessing_folder.mkdir(parents=True)

vocabulary_folder = dataset_path.joinpath("Vocabulary")
vocabulary_path = vocabulary_folder.joinpath(dataset_name + "_vocab.json")

print(f"Vocabulary path: {vocabulary_path}")

Datasets directory: /Users/paolo/PycharmProjects/Data
Recovering dataframe from pickle file /Users/paolo/PycharmProjects/Data/aclImdb/Dataframe/aclImdb.pkl
Preprocessing directory: /Users/paolo/PycharmProjects/Data/aclImdb/Preprocessing
Vocabulary path: /Users/paolo/PycharmProjects/Data/aclImdb/Vocabulary/aclImdb_vocab.json


Let's see what we've got

In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   file_id    50000 non-null  str  
 1   score      50000 non-null  int64
 2   sentiment  50000 non-null  int64
 3   split      50000 non-null  str  
 4   text       50000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 1.9 MB
None


In [4]:
df

,file_id,score,sentiment,split,text
0,4715,9,1,train,For a movie that gets no respect there sure ar...
1,12390,8,1,train,Bizarre horror movie filled with famous faces ...
2,8329,7,1,train,"A solid, if unremarkable film. Matthau, as Ein..."
3,9063,8,1,train,It's a strange feeling to sit alone in a theat...
4,3092,10,1,train,"You probably all already know this by now, but..."
...,...,...,...,...,...
49995,11513,1,0,test,With actors like Depardieu and Richard it is r...
49996,5409,1,0,test,If you like to get a couple of fleeting glimps...
49997,11187,1,0,test,When something can be anything you want it to ...
49998,9359,3,0,test,"I had heard good things about ""States of Grace..."


Let's see an example

In [5]:
print(df.loc[(df['file_id']=="0") & (df['split']=="train") & (df["sentiment"]==1),["text"]].values[0])

['Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High\'s satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers\' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I\'m here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn\'t!']


Let's define two variables to access the raw text data we're going to work with

In [6]:
df_train = df.loc[df['split'] == 'train']
df_test = df.loc[df['split'] == 'test']

train_texts = df_train.text.values
test_texts = df_test.text.values


print(f"train_texts/are instances of {type(train_texts)} and they contain {len(train_texts)} elements each")
print(train_texts[:3])

train_texts/are instances of <class 'pandas.arrays.StringArray'> and they contain 25000 elements each
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             'For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is a

## The Vocabulary
Pre-processing is an essential phase in the pipeline, leading to the representation of input documents by way of feature vectors, such as word embeddings. The first step is the definition of a vocabulary.
A vocabulary is a collection of atomic units of text occurring in a given dataset. It is possible to define a granularity for such atomic units (tokens): for example, characters, sub-words or words. For simplicity, we will refer to these tokens as "words." Representing the input requires that each input word is recognized and associated with an index in the vocabulary and a numeric representation in some feature space (embedding).

A vocabulary can be stored in a dictionary.





**Example**: Suppose you have the given toy corpus $D$: { "the cat is on the table" }

The dataset is comprised of only one sentence: "the cat is on the table". The corresponding vocabulary (a possible one) will be:

V = {0: 'the', 1: 'cat', 2: 'is', 3: 'on', 4: 'table'}

In this case, indexing follows word order, but it is not mandatory!

**Important:** The vocabulary should be produced following a well-defined deterministic procedure:
 make sure that the vocabulary creation routine always returns the same result!

In fact, a vocabulary is **exclusively defined** by the tokenization step you define!

$\rightarrow$ Characters, sub-words, words are examples of granularity levels.

### Preliminary pre-processing steps

Before vocabulary creation, we have to do a little bit of **text pre-processing** to avoid spurious data.

$\rightarrow$ Data quality is one of the crucial factors that lead to better performance.

$\rightarrow$ Models, even state-of-the-art ones, hardly achieve satisfying results if the dataset is very noisy.

$\rightarrow$ These pre-processing steps should be encoded in a precise routine to be applied to every input text (training data used to build models, and the test data used for performance evaluation)

**Types of pre-processing**: there are many pre-processing steps that we can consider, either general or quite task- specific.


*    **Text to lower**: casing usually doesn't affect our task, but in some scenarios, such as part-of-speech tagging, it is crucial.

*    **Replace special characters**: special characters are usually employed as variants of a single character like the spacing symbol ' '. In other cases (dates, etc..) special characters might have a specific meaning and should not be replaced.

*    **Text stripping**: it is important to filter out extra spaces to avoid unwanted distinctions between identical words, such as 'apple' and ' apple '.

Common pre-processing techniques include number replacing, lemmatization, stemming, removing stopwords, spell correction, acronyms merge and so on.

$\rightarrow$ If you are interested you can check [here](https://medium.com/swlh/text-normalization-7ecc8e084e31) and [here](https://www.kdnuggets.com/2019/04/text-preprocessing-nlp-machine-learning.html) some good blogs about the topic.

Let's look at some methods, with the NLTK library.

In [7]:
# preliminaries
import nltk
from nltk.tokenize import WhitespaceTokenizer

tokenizer = WhitespaceTokenizer()

# necessary for being able to tokenize
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /Users/paolo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/paolo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

NLTK includes resources like lists of stopwords

In [8]:
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer("english")

from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

print(f"{len(stop_words)} stop-words in English")
print(stop_words)

198 stop-words in English
{"you've", 'before', 'weren', "isn't", "i'd", 'during', 'once', "shouldn't", 'were', 'can', 'nor', 'me', 're', 'same', "wasn't", 'off', 'its', "that'll", 'after', 'are', 'been', "he'd", "you'll", 'do', "mustn't", 'ma', 'our', "i'll", 'she', 'more', 'the', 'ain', 'have', 'if', "it'd", 'or', 'who', 'than', 'ourselves', "haven't", 'having', "they're", "weren't", 'a', 'here', 'while', 'should', "he's", 'his', "they'll", 'too', 'o', 'own', 'yourself', 'they', 'just', 'most', "mightn't", 'was', 'hadn', 'how', 'other', "they'd", 'up', 'until', 'he', 'mustn', 'below', 'so', 'will', 'y', "i've", "couldn't", 'to', 'and', "needn't", "it'll", 'wasn', 'your', 'is', 'with', 'won', 'through', "aren't", 'over', 'out', 'am', 'herself', 'in', 'be', 'then', "we're", 'by', 'all', 'at', 'don', 'as', 'on', "doesn't", "won't", "we've", 'above', 'wouldn', 'being', 'such', "it's", 'very', "shan't", 'shouldn', 'which', 'them', "you're", 'haven', 'him', "she'll", 'himself', 'any', 'shan

[nltk_data] Downloading package stopwords to /Users/paolo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Preprocessing example: eliminate stopwords and apply stemming

In [9]:
def stem_text_no_stopword(text: str):
    words = [stemmer.stem(word) for word in tokenizer.tokenize(text) if word not in stop_words]
    return " ".join(words)

st_train_path = preprocessing_folder.joinpath(dataset_name + "_stemmed_train.json")
st_test_path = preprocessing_folder.joinpath(dataset_name + "_stemmed_test.json")

# Read from disk if possible
if st_train_path.exists() and st_test_path.exists():
    print("recovering stemmed texts from json files", st_train_path, "and", st_test_path)
    st_train_texts = []
    with st_train_path.open(mode='r') as f:
        st_train_texts = sj.load(f)
    st_test_texts = []
    with st_test_path.open(mode='r') as f:
        st_test_texts = sj.load(f)
else:
    print("preprocessing text")
    # Don't do this twice as it may take over a minute on standard hardware
    st_train_texts = [stem_text_no_stopword(text) for text in tqdm(train_texts)]
    st_test_texts = [stem_text_no_stopword(text) for text in tqdm(test_texts)]
    print(f"Saving stemmed train texts to {st_train_path}")
    with st_train_path.open(mode='w') as f:
        sj.dump(st_train_texts, f, indent=4)
    print(f"Saving stemmed test texts to {st_test_path}")
    with st_test_path.open(mode='w') as f:
        sj.dump(st_test_texts, f, indent=4)
    print("Saving completed!")


recovering stemmed texts from json files /Users/paolo/PycharmProjects/Data/aclImdb/Preprocessing/aclImdb_stemmed_train.json and /Users/paolo/PycharmProjects/Data/aclImdb/Preprocessing/aclImdb_stemmed_test.json


Difference between raw text and stemmed text. Notice that this stemming method includes lowercasing

In [10]:
print("raw input:\n", train_texts[0])
print()
print("stemmed:\n", st_train_texts[0])

raw input:
 For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt.

stemmed:
 for movi get respect sure lot memor quot list gem. imagin movi joe piscopo actual funny! maureen stapleton scene stealer. the moroni charact absolut scream. watch alan "the skipper" hale jr. polic sgt.


### Another preprocessing example: lemmatization
Lemmatization is an **alternative** to stemming (it's also significantly more time-consuming)

*''Lemmatization usually refers to use a vocabulary and morphological analysis of words to return the base or dictionary form of a word, known as **lemma**''*

<div align='right'>
(Introduction to Information Retrieval ~ C. D. Manning)
</div>

**Example**

*saw*, *seen*, *sees* $\rightarrow$ *see*

Lemmatization requires an ontology like WordNet to perform morphological analysis

In [11]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

def get_wordnet_key(pos_tag):
    if pos_tag.startswith('J'):
        return wordnet.ADJ
    elif pos_tag.startswith('V'):
        return wordnet.VERB
    elif pos_tag.startswith('N'):
        return wordnet.NOUN
    elif pos_tag.startswith('R'):
        return wordnet.ADV
    else:
        return 'n'

def lem_text(text: str):
    tokens = tokenizer.tokenize(text)
    tagged = pos_tag(tokens)
    words = [lemmatizer.lemmatize(word, get_wordnet_key(tag))
             for word, tag in tagged]
    return " ".join(words)

In [12]:
# Read from disk if possible
lem_train_path = preprocessing_folder.joinpath(dataset_name + "_lemmatized_train.json")
lem_test_path = preprocessing_folder.joinpath(dataset_name + "_lemmatized_test.json")

if lem_train_path.exists() and lem_test_path.exists():
    print("recovering lemmatized texts from json files", lem_train_path, "and", lem_test_path)
    lem_train_texts = []
    with lem_train_path.open(mode='r') as f:
        lem_train_texts = sj.load(f)
    lem_test_texts = []
    with lem_test_path.open(mode='r') as f:
        lem_test_texts = sj.load(f)
else:
    print("preprocessing text")
    # I don't want to do this twice as it may take about 6 minutes on standard hardware
    nltk.download('omw-1.4')
    nltk.download('wordnet')
    nltk.download('averaged_perceptron_tagger')
    nltk.download('averaged_perceptron_tagger_eng')

    lemmatizer = WordNetLemmatizer()
    lem_train_texts = [lem_text(text) for text in tqdm(train_texts)]
    lem_test_texts = [lem_text(text) for text in tqdm(test_texts)]
    print(f"Saving lemmatized train texts to {lem_train_path}")
    with lem_train_path.open(mode='w') as f:
        sj.dump(lem_train_texts, f, indent=4)
    print(f"Saving lemmatized test texts to {lem_test_path}")
    with lem_test_path.open(mode='w') as f:
        sj.dump(lem_test_texts, f, indent=4)
    print("Saving completed!")

recovering lemmatized texts from json files /Users/paolo/PycharmProjects/Data/aclImdb/Preprocessing/aclImdb_lemmatized_train.json and /Users/paolo/PycharmProjects/Data/aclImdb/Preprocessing/aclImdb_lemmatized_test.json


Difference between raw text and lemmatized tesxt

In [13]:
print("raw input:\n", train_texts[0])
print()
print("stemmed:\n", st_train_texts[0])
print()
print("lemmatized:\n", lem_train_texts[0])

raw input:
 For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt.

stemmed:
 for movi get respect sure lot memor quot list gem. imagin movi joe piscopo actual funny! maureen stapleton scene stealer. the moroni charact absolut scream. watch alan "the skipper" hale jr. polic sgt.

lemmatized:
 For a movie that get no respect there sure be a lot of memorable quote list for this gem. Imagine a movie where Joe Piscopo be actually funny! Maureen Stapleton be a scene stealer. The Moroni character be an absolute scream. Watch for Alan "The Skipper" Hale jr. a a police Sgt.


## Preprocessing pipeline example: clean up text and tokenize


In [14]:
REPLACE_BY_SPACE_RE = re.compile(r'[/(){}\[\]\|@,;]')
GOOD_SYMBOLS_RE = re.compile(r'[^0-9a-z #+_]')
try:
    STOPWORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    STOPWORDS = set(stopwords.words('english'))

In [15]:
def lower(text: str) -> str:
    """
    Transforms given text to lower case.
    """
    return text.lower()

def replace_special_characters(text: str) -> str:
    """
    Replaces special characters with spacing character
    """
    return REPLACE_BY_SPACE_RE.sub(' ', text)

def replace_br(text: str) -> str:
    """
    Replaces br characters
    """
    return text.replace('br', '')

In [16]:
def filter_out_uncommon_symbols(text: str) -> str:
    """
    Removes any special character that is not
    in the good symbols list (check regular expression)
    """
    return GOOD_SYMBOLS_RE.sub('', text)

def remove_stopwords(text: str) -> str:
    return ' '.join([x for x in text.split()
                     if x and x not in STOPWORDS])

def strip_text(text: str) -> str:
    """
    Removes any left or right spacing
    (including carriage return) from text.
    """
    return text.strip()

Once we have defined all the steps that we consider relevant for our task, we assemble them in a pre-processing pipeline. The `reduce` methods is useful.

In [17]:
PREPROCESSING_PIPELINE = [
                          lower,
                          replace_special_characters,
                          replace_br,
                          filter_out_uncommon_symbols,
                          remove_stopwords,
                          strip_text
                          ]

def text_prepare(
    text: str,
    filter_methods: List[Callable[[str], str]] = None
) -> str:
    """
    Applies a list of pre-processing functions in sequence (reduce).
    Note that the order is important here!
    """
    filter_methods = filter_methods \
    if filter_methods is not None else PREPROCESSING_PIPELINE
    return reduce(lambda txt, f: f(txt), filter_methods, text)

In [18]:
def apply_preprocessing_pipeline(
        df : pd.DataFrame
):
    print('Pre-processing text...')
    print()
    print(f'[Debug] Before:\n{df.text.values[20][:203]}...')
    print()

    # Replace each sentence with its pre-processed version
    df['text'] = df['text'].apply(lambda txt: text_prepare(txt))

    print(f'[Debug] After:\n{df.text.values[20][:203]}...')
    print()
    print("Pre-processing completed!")

### Creating a vocabulary


Once the text has been pre-processed, space splitting should work correctly.
We can see how to build a vocabulary and perform some sanity checks.

$\rightarrow$ Bear in mind that some packages offers tools for automatic vocabulary creation, such as [NLTK](https://www.nltk.org/api/nltk.tokenize.html) and Keras (check [keras.preprocessing.text.Tokenizer](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer)).


**Note**: In this case, the vocabulary will start from index equal to 1, since the 0 slot is reserved to padding token.

In [19]:
def build_vocabulary(
    df: pd.DataFrame
) -> (Dict[int, str], Dict[str, int], List[str]):
    """
    Given a dataset, builds the corresponding word vocabulary.

    :param df: dataset from which we want to build
    the word vocabulary (pandas.DataFrame)
    :return:
      - word vocabulary: vocabulary index to word
      - inverse word vocabulary: word to vocabulary index
      - word listing: set of unique terms that build up the vocabulary
    """
    idx_to_word = OrderedDict()
    word_to_idx = OrderedDict()

    curr_idx = 0
    for sentence in tqdm(df.text.values):
        tokens = sentence.split()
        for token in tokens:
            if token not in word_to_idx:
                word_to_idx[token] = curr_idx
                idx_to_word[curr_idx] = token
                curr_idx += 1

    word_listing = list(idx_to_word.values())
    return idx_to_word, word_to_idx, word_listing


### Note
Define **intermediary tests** for your code in order to inspect data and to assess the correctness of your code!

$\rightarrow$ You don't want to **re-run** huge and time-consuming experiments due to early pipeline errors!

In [20]:
def evaluate_vocabulary(
    idx_to_word: Dict[int, str],
    word_to_idx: Dict[str, int],
    word_listing: List[str],
    df: pd.DataFrame,
    check_default_size: bool = False
):
    assert len(idx_to_word) == len(word_to_idx)
    assert len(idx_to_word) == len(word_listing)

    for i in tqdm(range(0, len(idx_to_word))):
        assert idx_to_word[i] in word_to_idx
        assert word_to_idx[idx_to_word[i]] == i

    _, _, first_word_listing = build_vocabulary(df)
    _, _, second_word_listing = build_vocabulary(df)
    assert first_word_listing == second_word_listing

    toy_df = pd.DataFrame.from_dict({
        'text': ["all that glitters is not gold",
                 "all in all i like this assignment"]
    })
    _, _, toy_word_listing = build_vocabulary(toy_df)
    toy_valid_vocabulary = set(' '.join(toy_df.text.values).split())
    assert set(toy_word_listing) == toy_valid_vocabulary

### Saving the vocabulary
Generally speaking, it is a good idea to save the dictionary in clear format, for example JSON.

$\rightarrow$ In this way you can quickly check for errors or useful words.

In [21]:
if vocabulary_path.exists():
    #recover from jason file
    print("recovering vocab from json file", vocabulary_path)
    with vocabulary_path.open(mode='r') as f:
        word_to_idx = OrderedDict()
        # read word_to_idx: Dict[str, int] from file
        word_to_idx=sj.load(f)
        # construct idx_to_word: Dict[int, str] from word_to_idx
        idx_to_word = OrderedDict()
        for word, index in tqdm(word_to_idx.items()):
            idx_to_word[index] = word
        # construct word_listing: List[str] from word_to_idx
        word_listing = list(idx_to_word.values())
    print("vocab recovered")
else:
    if not vocabulary_folder.exists():
        print(f"creating folder {vocabulary_folder}")
        vocabulary_folder.mkdir(parents=True)
    # pre-process text
    apply_preprocessing_pipeline(df)
    # create vocabulary
    print("creating vocabulary")
    idx_to_word, word_to_idx, word_listing = build_vocabulary(df)
    print(f'[Debug] Index -> Word vocabulary size: {len(idx_to_word)}')
    print(f'[Debug] Word -> Index vocabulary size: {len(word_to_idx)}')

    print("Vocabulary evaluation...")
    evaluate_vocabulary(idx_to_word, word_to_idx, word_listing, df)
    print("Evaluation completed!")

    print(f"Saving vocabulary to {vocabulary_path}")
    with vocabulary_path.open(mode='w') as f:
        sj.dump(word_to_idx, f, indent=4)
    print("Saving completed!")


some_words = [(idx_to_word[idx], idx) for idx in np.arange(10) + 1]
print(f'[Debug] Some words: {some_words}')

recovering vocab from json file /Users/paolo/PycharmProjects/Data/aclImdb/Vocabulary/aclImdb_vocab.json


100%|██████████| 152534/152534 [00:00<00:00, 5353174.69it/s]

vocab recovered
[Debug] Some words: [('gets', np.int64(1)), ('respect', np.int64(2)), ('sure', np.int64(3)), ('lot', np.int64(4)), ('memorable', np.int64(5)), ('quotes', np.int64(6)), ('listed', np.int64(7)), ('gem', np.int64(8)), ('imagine', np.int64(9)), ('joe', np.int64(10))]


#### Homework 📖


Spend some time at checking the built vocabulary!

In many tasks (e.g., sequential tagging, embedding analysis, etc...), the way text is tokenized is crucial!

## Vectorization with scikit-learn
Once the text has been pre-processed and tokenized, we can extract the features that represent each document. This step is also called vectorization, since for each input document we need a feature vector.
<center>
    <img src="images/inputs_outputs.png" alt="inputs and outputs" />
</center>

<center>
$\downarrow$ vectorization $\downarrow$
</center>

<center>
    <img src="images/inputs_outputs_features.png" alt="features and outputs" />
</center>

### Bag-of-Words Encoding

The [scikit-learn](https://scikit-learn.org/stable/) library offers *CountVectorizer*, a class that performs both tokenization and the creation of a Bag-of-Words (BoW) representation of the documents in a corpus.

Notice that this method includes a default tokenizer, which uses a default token_pattern: `r"(?u)\\b\\w\\w+\\b"`, hence the vocabulary is automatically determined by fitting the vectorized to the training data, and stored in `vectorizer.vocabulary_`.

The vectorizer can be invoked with several arguments, for example to use a different token_pattern (check the documentation).

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [23]:
vectorizer =  CountVectorizer()
X_train_bow = vectorizer.fit_transform(tqdm(train_texts,position=0))

print(f"{X_train_bow.shape[1]} features")

100%|██████████| 25000/25000 [00:00<00:00, 26682.12it/s]


74849 features


In [24]:
print(f"First document's features:")
print(X_train_bow[0])

First document's features:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 41 stored elements and shape (1, 74849)>
  Coords	Values
  (0, 25450)	3
  (0, 44147)	2
  (0, 66322)	1
  (0, 27312)	1
  (0, 45805)	1
  (0, 55262)	1
  (0, 66432)	1
  (0, 64589)	1
  (0, 4124)	1
  (0, 39486)	1
  (0, 46680)	1
  (0, 42134)	1
  (0, 53071)	1
  (0, 38985)	1
  (0, 66562)	1
  (0, 27073)	1
  (0, 32667)	1
  (0, 72757)	1
  (0, 35415)	1
  (0, 50102)	1
  (0, 34585)	3
  (0, 1670)	1
  (0, 26385)	1
  (0, 41471)	1
  (0, 62846)	1
  (0, 57873)	1
  (0, 63002)	1
  (0, 66339)	2
  (0, 43900)	1
  (0, 11615)	1
  (0, 3167)	1
  (0, 1316)	1
  (0, 58249)	1
  (0, 72246)	1
  (0, 2464)	1
  (0, 60644)	1
  (0, 29499)	1
  (0, 35627)	1
  (0, 4465)	1
  (0, 50646)	1
  (0, 59148)	1




It's a sparse format, which shows only non-zero features (counters). See [scipy.sparse](https://docs.scipy.org/doc/scipy/reference/sparse.html) for more info. To see its matrix visualization you can use method toarray()

Each feature corresponds to a vocabulary entry with non-zero count (one or more tokens). Let's look at detected tokens.

In [25]:
vocab = vectorizer.vocabulary_
print(f"Vocabulary of {type(vocab)} and size {len(vocab)}.")
#a_word = input("Enter a word: ")
a_word = "beautiful"
try:
    print(f"the word '{a_word}' is vocabulary entry {vocab[a_word]}")
except:
    print(f"The word '{a_word}' is not in the vocabulary")

Vocabulary of <class 'dict'> and size 74849.
the word 'beautiful' is vocabulary entry 6440


These are the non-zero features of document 0:

In [26]:
print(train_texts[0], " ", len(tokenizer.tokenize(train_texts[0])), "words")
print()
print(X_train_bow[0].indices, " ", len(X_train_bow[0].indices),"non-zero features")

For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt.   52 words

[25450 44147 66322 27312 45805 55262 66432 64589  4124 39486 46680 42134
 53071 38985 66562 27073 32667 72757 35415 50102 34585  1670 26385 41471
 62846 57873 63002 66339 43900 11615  3167  1316 58249 72246  2464 60644
 29499 35627  4465 50646 59148]   41 non-zero features


These are the words they correspond to:

In [27]:
def show_vocab_words(document):
    for idx in X_train_bow[document].indices:
        # getting the key ba value in a dictionary is non-obvious. On possibility is to separate the dictionary's values in a list, find the position of idx, and get the key at that position. If this kind of look-up is something needed frequently, it would be better to create a second dictionary where keys and values switch places (possible, since values are unique too)
        print(f"{idx}: {list(vocab.keys())[list(vocab.values()).index(idx)]}")

show_vocab_words(0)

25450: for
44147: movie
66322: that
27312: gets
45805: no
55262: respect
66432: there
64589: sure
4124: are
39486: lot
46680: of
42134: memorable
53071: quotes
38985: listed
66562: this
27073: gem
32667: imagine
72757: where
35415: joe
50102: piscopo
34585: is
1670: actually
26385: funny
41471: maureen
62846: stapleton
57873: scene
63002: stealer
66339: the
43900: moroni
11615: character
3167: an
1316: absolute
58249: scream
72246: watch
2464: alan
60644: skipper
29499: hale
35627: jr
4465: as
50646: police
59148: sgt


We use our fitted Vectorizer to transform test texts as well. **Be sure** to invoke ``vectorizer.transform()`` and **not** ``vectorizer.fit_transform()``.


In [28]:
X_test_bow = vectorizer.transform(tqdm(test_texts,position=0))

100%|██████████| 25000/25000 [00:00<00:00, 26475.30it/s]


Another possible vectorization is by tf-idf reweighting, using `TfidfVectorizer()` instead of `CountVectorizer()`


In [29]:
vectorizer =  TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(tqdm(train_texts,position=0))
X_test_tfidf = vectorizer.transform(tqdm(test_texts,position=0))

print(f"{X_train_tfidf.shape[1]} features")

100%|██████████| 25000/25000 [00:00<00:00, 25416.06it/s]


74849 features


It is possible to run BoW/TdfIdf Vectorization after other preliminary preprocessing steps like lowercasing, stemming and lemmatization. Eventually, we can run ablation studies to decide which steps in the pipeline benefit the downstream task by comparing classifier performance.

In [30]:
def extract_features (vectorizer_class, raw_train_texts, raw_test_texts):
    vectorizer = vectorizer_class
    X_train = vectorizer.fit_transform(tqdm(raw_train_texts, position=0))
    X_test = vectorizer.transform(tqdm(raw_test_texts, position=0))
    print(f"extracted {X_train.shape[1]} features")
    return X_train, X_test

Example: Extract BoW features with stemming

In [31]:
st_X_train_bow, st_X_test_bow = extract_features(CountVectorizer(), st_train_texts, st_test_texts)

100%|██████████| 25000/25000 [00:00<00:00, 40186.20it/s]


extracted 70362 features


In this case, we don't need to save BoW/Tfidf features to disk as it takes little time to extract them

# Feature selection
It may be preferable or necessary to operate with a smaller vocabulary. We can select a subset words, for example based on their frequency in the training corpus, by eliminating words that are too frequent (like stopwords) or too infrequent (whereby no meaningful statistics can be learned), or by looking at the correlation between features and target class, and keeping only the features with a more significant correlation.

## Recursive Feature Elimination
We can use Recursive Feature Elimination (RFE) for feature selection. RFE involves training a model on a subset of the features, and then iteratively removing the least important features one by one until we are left with the desired number of features. However, if run 1 feature at a time, it is not scalable with a feature space of tens of thousands (the vocab size). Hence, we can first run a **Chi-Squared filter**, to score all features instantly, and keep only the top 3000 features. Then, we run RFE with a step of 0.10, meaning that we drop 10% of the features at every step, until we reach the desider number of features, say 50. This number can be a hyperparameter, to be tuned on some validation set.

First of all, we need the labels of the training set

In [32]:
Y_train = df_train.sentiment.values

The, we run a Chi-Squared test using the `SelectKBest` method from `sklearn.feature_selection` for example using the tfidf features extracted from the original text without any specific pre-processing pipeline. We obtain 3000 features.

In [33]:
# run chi-squared test
from sklearn.feature_selection import SelectKBest, chi2
pre_selector = SelectKBest(score_func=chi2, k=3000)
X_train_filtered = pre_selector.fit_transform(X_train_tfidf, Y_train)

print(f"Original shape: {X_train_tfidf.shape}")          # e.g., (25000, 74849)
print(f"Filtered shape: {X_train_filtered.shape}")  # (25000, 3000)

Original shape: (25000, 74849)
Filtered shape: (25000, 3000)


Now, in `X_train_filtered` we have vectors of 3K features, selected only based on their correlation with the labels, by only using the training set.

It is interesting to see how the features have been selected, based on their scores. We look into the fitted `pre_selector`.

In [34]:
chi2_scores = pre_selector.scores_
p_values = pre_selector.pvalues_
feature_names = vectorizer.get_feature_names_out()

# these are the indices of the features that have been selected
pre_selected_feature_indices = pre_selector.get_support(indices=True)
print(f"These are the indices of the {pre_selected_feature_indices.shape[0]} selected features:",pre_selected_feature_indices)


These are the indices of the 3000 selected features: [   17    85   100 ... 74735 74771 74780]


In [35]:
# Create a clean DataFrame to inspect the results
feature_importance = pd.DataFrame({
    'Feature': feature_names, # the vocabulary item
    'Chi2_Score': chi2_scores, # correlation with label
    'P_Value': p_values # p-values of feature scores
}).sort_values(by='Chi2_Score', ascending=False)

These are the 10 top-scoring features out of 3K selected

In [36]:
feature_importance.head(10)

,Feature,Chi2_Score,P_Value
5502,bad,137.791532,8.094334e-32
73700,worst,112.677997,2.538236e-26
72229,waste,83.307099,7.024688e-20
5266,awful,72.683013,1.522388e-17
28545,great,62.267989,2.997582e-15
66150,terrible,58.551624,1.980261e-14
31669,horrible,49.726596,1.767339e-12
73683,worse,49.280061,2.219066e-12
8512,boring,48.767406,2.881896e-12
22912,excellent,47.542635,5.382058e-12


Now, let's reduce the features even further to `N_features_to_select` total, using RFE on the filtered features.
For RFE, we use an SVC classifier with a linear kernel.

In [37]:
# run RFE on subset
from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC

N_features_to_select = 1000

# Use RFE to select the top 10 features
selector = RFE(LinearSVC(), n_features_to_select=N_features_to_select, step=0.10)
selector.fit(X_train_filtered, Y_train)

final_features_ids = selector.get_feature_names_out()
# Print the selected features
print(final_features_ids)
print(f"{len(final_features_ids)} features")

['x1' 'x3' 'x9' 'x23' 'x25' 'x30' 'x31' 'x36' 'x46' 'x51' 'x52' 'x61'
 'x67' 'x74' 'x75' 'x77' 'x79' 'x83' 'x84' 'x89' 'x90' 'x92' 'x93' 'x96'
 'x97' 'x98' 'x99' 'x102' 'x108' 'x110' 'x111' 'x113' 'x119' 'x120' 'x128'
 'x134' 'x135' 'x137' 'x138' 'x139' 'x141' 'x143' 'x149' 'x150' 'x152'
 'x156' 'x157' 'x160' 'x161' 'x166' 'x167' 'x168' 'x170' 'x176' 'x178'
 'x184' 'x185' 'x186' 'x187' 'x194' 'x195' 'x199' 'x200' 'x201' 'x206'
 'x209' 'x215' 'x220' 'x223' 'x224' 'x225' 'x227' 'x235' 'x236' 'x238'
 'x243' 'x244' 'x245' 'x247' 'x249' 'x254' 'x258' 'x259' 'x263' 'x269'
 'x270' 'x274' 'x275' 'x276' 'x277' 'x281' 'x282' 'x286' 'x287' 'x290'
 'x291' 'x293' 'x296' 'x298' 'x299' 'x300' 'x301' 'x302' 'x303' 'x307'
 'x308' 'x313' 'x316' 'x318' 'x320' 'x326' 'x327' 'x328' 'x335' 'x340'
 'x343' 'x344' 'x346' 'x347' 'x353' 'x356' 'x367' 'x371' 'x374' 'x375'
 'x377' 'x378' 'x388' 'x389' 'x390' 'x394' 'x397' 'x399' 'x401' 'x408'
 'x410' 'x413' 'x421' 'x424' 'x426' 'x427' 'x432' 'x439' 'x446' 'x447'
 

In [38]:
selected_features = []

for feature in final_features_ids:
    entry = feature_importance.iloc[int(feature[1:])]
    dataframe_row = {
                        "idx": int(feature[1:]),
                        "Feature": entry.Feature,
                        "Chi2_Score": entry.Chi2_Score,
                        "P_Value": entry.P_Value
                    }
    selected_features.append(dataframe_row)

selected_features_fd = pd.DataFrame(selected_features).sort_values(by="Chi2_Score",ascending=False)
selected_features_fd[:10]

,idx,Feature,Chi2_Score,P_Value
0,1,worst,112.677997,2.538236e-26
1,3,awful,72.683013,1.522388e-17
2,9,excellent,47.542635,5.382058e-12
3,23,best,29.762435,4.883638e-08
4,25,loved,27.751140,1.379669e-07
5,30,superb,25.253922,5.025728e-07
6,31,even,25.120283,5.386332e-07
7,36,plot,24.150836,8.907747e-07
8,46,save,20.018563,7.669403e-06
9,51,unless,19.514553,9.983616e-06


How well did we do?

Let's consider the three sets of features: 75K (`X_train_tfidf`/`X_test_tfidf`), 3K (`X_train_filtered`/`X_test_filtered`) and 1K (`X_train_selected`/`X_test_selected`).

In [39]:
# we define an svc_classifier
svc_classifier = LinearSVC()

# 75K features
# X_train_tfidf contains the 75K features of the train set obtained using vectorizer.fit_transform
# X_test_tfidf contains the 75K features of the test set by applying the fitted vectorizer

# 3K features
# X_train_filtered contains the features obtained as a result of fitting the pre_selector on the training data
# we apply the same pre_selector on the 75K features of the test set
X_test_filtered = pre_selector.transform(X_test_tfidf)

# 1K features, train/test set: we apply the selector, which was fitted on the training set
X_train_selected = selector.transform(X_train_filtered)
X_test_selected = selector.transform(X_test_filtered)

# finally, we need the test set labels to measure the performance
Y_test = df_test.sentiment.values

For each feature configuration, let's fit an SVC classifier using the train features/labels, and then predict using the test set features

In [40]:
# first case: 75K features
svc_classifier.fit(X_train_tfidf, Y_train)
Y_pred_test_tfidf = svc_classifier.predict(X_test_tfidf)

# second case: 3K features
svc_classifier.fit(X_train_filtered, Y_train)
Y_pred_test_filtered = svc_classifier.predict(X_test_filtered)

# third case: 1K features
svc_classifier.fit(X_train_selected, Y_train)
Y_pred_test_selected = svc_classifier.predict(X_test_selected)

Les us now evaluate based on F1

In [41]:
from sklearn.metrics import classification_report

def evaluate_classification(y_true, y_pred):
    report = classification_report(y_true,
                                   y_pred,
                                   target_names=['neg', 'pos'],
                                   zero_division=0.0,
                                   digits = 4)
    print(report)

print("75K features")
evaluate_classification(Y_test, Y_pred_test_tfidf)

print("3K features")
evaluate_classification(Y_test, Y_pred_test_filtered)

print("1K features")
evaluate_classification(Y_test, Y_pred_test_selected)

75K features
              precision    recall  f1-score   support

         neg     0.8690    0.8883    0.8786     12500
         pos     0.8858    0.8661    0.8758     12500

    accuracy                         0.8772     25000
   macro avg     0.8774    0.8772    0.8772     25000
weighted avg     0.8774    0.8772    0.8772     25000

3K features
              precision    recall  f1-score   support

         neg     0.8808    0.8846    0.8827     12500
         pos     0.8841    0.8803    0.8822     12500

    accuracy                         0.8824     25000
   macro avg     0.8824    0.8824    0.8824     25000
weighted avg     0.8824    0.8824    0.8824     25000

1K features
              precision    recall  f1-score   support

         neg     0.8815    0.8705    0.8759     12500
         pos     0.8721    0.8830    0.8775     12500

    accuracy                         0.8767     25000
   macro avg     0.8768    0.8767    0.8767     25000
weighted avg     0.8768    0.8767    

| Features    | F1-macro | $\Delta$ |
| ---         | ---      | ---      |
| tfidf-75K  | .8772    | - |
| tfdidf-3K | **.8824**    | .0052 |
| tfidf-1K  | .8767    | -.0005 |

A larger vocabulary does not necessarily mean a better one!

Besides: _are these differences significant?_